In [ ]:
import os
import random
import json
import requests
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Step 1: Pick 10 random images from the folder
def pick_random_images(folder_path, num_images=10):
    images = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    return random.sample(images, min(num_images, len(images)))

# Step 2: Retrieve IMDb data for each image using the IMDb ID
def get_imdb_data(imdb_id):
    url = f"https://www.imdb.com/title/{imdb_id}/"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    script = soup.find('script', type='application/ld+json')
    if script:
        return json.loads(script.string)
    return None

# Step 3: Save the data to a CSV file
def save_to_csv(data, csv_file):
    df = pd.DataFrame(data)
    df.to_csv(csv_file, index=False)

# Step 4: Perform PCA on the data
def perform_pca(data):
    # Convert data to numerical format (example: using aggregateRating)
    ratings = [float(item['aggregateRating']['ratingValue']) for item in data]
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform([[rating] for rating in ratings])
    return pca_result

# Step 5: Visualize the PCA results
def visualize_pca(pca_result, titles):
    plt.figure(figsize=(10, 6))
    plt.scatter(pca_result[:, 0], pca_result[:, 1])
    for i, title in enumerate(titles):
        plt.text(pca_result[i, 0], pca_result[i, 1], title)
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.title('PCA of IMDb Ratings')
    plt.show()

# Main function
def main(folder_path, csv_file):
    # Step 1: Pick 10 random images
    random_images = pick_random_images(folder_path)
    imdb_ids = [os.path.splitext(img)[0] for img in random_images]

    # Step 2: Retrieve IMDb data
    imdb_data = []
    for imdb_id in imdb_ids:
        data = get_imdb_data(imdb_id)
        if data:
            imdb_data.append(data)

    # Step 3: Save to CSV
    save_to_csv(imdb_data, csv_file)

    # Step 4: Perform PCA
    pca_result = perform_pca(imdb_data)

    # Step 5: Visualize PCA
    titles = [data['name'] for data in imdb_data]
    visualize_pca(pca_result, titles)

if __name__ == "__main__":
    folder_path = "path/to/your/image/folder"
    csv_file = "imdb_data.csv"
    main(folder_path, csv_file)